## 1. 라이브러리 및 환경 설정

In [38]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from catboost import CatBoostRegressor

## 2. 데이터 로드

In [39]:
train = pd.read_csv('./data/train.csv')
test = pd.read_csv('./data/test.csv')
layout = pd.read_csv('./data/layout_info.csv')

if 'layout_type' not in train.columns:
    train = train.merge(layout, on='layout_id', how='left')
    test = test.merge(layout, on='layout_id', how='left')

print(f"학습 데이터 크기: {train.shape}")
print(f"테스트 데이터 크기: {test.shape}")

학습 데이터 크기: (250000, 108)
테스트 데이터 크기: (50000, 107)


In [40]:
# scenario별 행 개수 확인
print(train['scenario_id'].value_counts().describe())
print(test['scenario_id'].value_counts().describe())

# 같은 scenario 내에서 ID가 증가하는지 샘플 확인
sample_scenarios = train['scenario_id'].drop_duplicates().head(5).tolist()

for s in sample_scenarios:
    tmp = train.loc[train['scenario_id'] == s, ['scenario_id', 'ID']].copy()
    print(f"\n[scenario_id = {s}]")
    print(tmp.head(10))
    print("ID 오름차순 여부:", tmp['ID'].is_monotonic_increasing)

count    10000.0
mean        25.0
std          0.0
min         25.0
25%         25.0
50%         25.0
75%         25.0
max         25.0
Name: count, dtype: float64
count    2000.0
mean       25.0
std         0.0
min        25.0
25%        25.0
50%        25.0
75%        25.0
max        25.0
Name: count, dtype: float64

[scenario_id = SC_07598]
  scenario_id            ID
0    SC_07598  TRAIN_000000
1    SC_07598  TRAIN_000001
2    SC_07598  TRAIN_000002
3    SC_07598  TRAIN_000003
4    SC_07598  TRAIN_000004
5    SC_07598  TRAIN_000005
6    SC_07598  TRAIN_000006
7    SC_07598  TRAIN_000007
8    SC_07598  TRAIN_000008
9    SC_07598  TRAIN_000009
ID 오름차순 여부: True

[scenario_id = SC_05443]
   scenario_id            ID
25    SC_05443  TRAIN_000025
26    SC_05443  TRAIN_000026
27    SC_05443  TRAIN_000027
28    SC_05443  TRAIN_000028
29    SC_05443  TRAIN_000029
30    SC_05443  TRAIN_000030
31    SC_05443  TRAIN_000031
32    SC_05443  TRAIN_000032
33    SC_05443  TRAIN_000033
34    SC_0544

In [41]:
# ===== 결측 feature =====
missing_ratio = train.isnull().mean().sort_values(ascending=False)
top_missing_cols = missing_ratio[missing_ratio > 0].head(10).index.tolist()

print("결측 컬럼:", top_missing_cols)

for col in top_missing_cols:
    train[f'{col}_isna'] = train[col].isna().astype(int)
    test[f'{col}_isna'] = test[col].isna().astype(int)

train['num_missing'] = train.isnull().sum(axis=1)
test['num_missing'] = test.isnull().sum(axis=1)

train = train.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
test = test.sort_values(['scenario_id', 'ID']).reset_index(drop=True)

def add_time_features(df):
    df = df.copy()

    group_col = 'scenario_id'

    # lag / diff용
    base_cols = [
        'order_inflow_15m',
        'congestion_score',
        'robot_charging',
        'battery_mean',
        'pack_utilization'
    ]

    for col in base_cols:
        # lag
        df[f'{col}_lag1'] = df.groupby(group_col)[col].shift(1)
        df[f'{col}_lag2'] = df.groupby(group_col)[col].shift(2)

        # diff
        df[f'{col}_diff1'] = df[col] - df.groupby(group_col)[col].shift(1)

    # ===== rolling mean / std 추가 =====
    rolling_mean_cols = [
        'order_inflow_15m',
        'congestion_score',
        'robot_charging'
    ]

    for col in rolling_mean_cols:
        df[f'{col}_rolling_mean_2'] = (
            df.groupby(group_col)[col]
              .transform(lambda x: x.shift(1).rolling(2, min_periods=1).mean())
        )

        df[f'{col}_rolling_mean_3'] = (
            df.groupby(group_col)[col]
              .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
        )

    # battery_mean은 std까지 추가
    df['battery_mean_rolling_std_3'] = (
        df.groupby(group_col)['battery_mean']
          .transform(lambda x: x.shift(1).rolling(3, min_periods=1).std())
    )

    return df

train = add_time_features(train)
test = add_time_features(test)

train = train.fillna(0)
test = test.fillna(0)

결측 컬럼: ['avg_recovery_time', 'congestion_score', 'avg_charge_wait', 'battery_mean', 'charge_efficiency_pct', 'battery_cycle_count_avg', 'fleet_age_months_avg', 'robot_calibration_score', 'unique_sku_15m', 'staging_area_util']


In [ ]:
print([col for col in train.columns if 'layout_type' in col])
print([col for col in test.columns if 'layout_type' in col])

# ===== 파생 변수 생성 =====
train['order_per_robot'] = train['order_inflow_15m'] / (train['robot_active'] + 1)
test['order_per_robot'] = test['order_inflow_15m'] / (test['robot_active'] + 1)

train['charge_pressure'] = train['robot_charging'] / (train['charger_count'] + 1)
test['charge_pressure'] = test['robot_charging'] / (test['charger_count'] + 1)

train['congestion_per_robot'] = train['congestion_score'] / (train['robot_active'] + 1)
test['congestion_per_robot'] = test['congestion_score'] / (test['robot_active'] + 1)

# ===== 기본 파생 변수 =====
train['order_per_robot'] = train['order_inflow_15m'] / (train['robot_active'] + 1)
test['order_per_robot'] = test['order_inflow_15m'] / (test['robot_active'] + 1)

train['charge_pressure'] = train['robot_charging'] / (train['charger_count'] + 1)
test['charge_pressure'] = test['robot_charging'] / (test['charger_count'] + 1)

train['congestion_per_robot'] = train['congestion_score'] / (train['robot_active'] + 1)
test['congestion_per_robot'] = test['congestion_score'] / (test['robot_active'] + 1)

# ===== 운영 비율 / 병목 feature 확장 =====
train['orders_per_pack_station'] = train['order_inflow_15m'] / (train['pack_station_count'] + 1)
test['orders_per_pack_station'] = test['order_inflow_15m'] / (test['pack_station_count'] + 1)

train['active_per_aisle_width'] = train['robot_active'] / (train['aisle_width_avg'] + 1)
test['active_per_aisle_width'] = test['robot_active'] / (test['aisle_width_avg'] + 1)

train['congestion_per_aisle'] = train['congestion_score'] / (train['aisle_width_avg'] + 1)
test['congestion_per_aisle'] = test['congestion_score'] / (test['aisle_width_avg'] + 1)

train['charging_per_robot'] = train['robot_charging'] / (train['robot_active'] + 1)
test['charging_per_robot'] = test['robot_charging'] / (test['robot_active'] + 1)

train['low_battery_pressure'] = train['low_battery_ratio'] * train['robot_active']
test['low_battery_pressure'] = test['low_battery_ratio'] * test['robot_active']

train['pack_pressure'] = train['pack_utilization'] / (train['pack_station_count'] + 1)
test['pack_pressure'] = test['pack_utilization'] / (test['pack_station_count'] + 1)

train['intersection_pressure'] = train['congestion_score'] * train['intersection_count']
test['intersection_pressure'] = test['congestion_score'] * test['intersection_count']

train['robot_density'] = train['robot_total'] / (train['floor_area_sqm'] + 1)
test['robot_density'] = test['robot_total'] / (test['floor_area_sqm'] + 1)

train['charger_per_robot'] = train['charger_count'] / (train['robot_total'] + 1)
test['charger_per_robot'] = test['charger_count'] / (test['robot_total'] + 1)

train['emergency_complexity'] = train['emergency_exit_count'] / (train['floor_area_sqm'] + 1)
test['emergency_complexity'] = test['emergency_exit_count'] / (test['floor_area_sqm'] + 1)

# ===== interaction feature (곱) =====
train['congestion_x_robot_active'] = train['congestion_score'] * train['robot_active']
test['congestion_x_robot_active'] = test['congestion_score'] * test['robot_active']

train['order_inflow_x_congestion'] = train['order_inflow_15m'] * train['congestion_score']
test['order_inflow_x_congestion'] = test['order_inflow_15m'] * test['congestion_score']

train['low_battery_x_robot_active'] = train['low_battery_ratio'] * train['robot_active']
test['low_battery_x_robot_active'] = test['low_battery_ratio'] * test['robot_active']

train['pack_util_x_order_inflow'] = train['pack_utilization'] * train['order_inflow_15m']
test['pack_util_x_order_inflow'] = test['pack_utilization'] * test['order_inflow_15m']

# train/test 동일 기준으로 layout_type 인코딩
combined = pd.concat([train['layout_type'], test['layout_type']], axis=0).astype(str)
mapping = {v: i for i, v in enumerate(combined.unique())}

train['layout_type'] = train['layout_type'].astype(str).map(mapping)
test['layout_type'] = test['layout_type'].astype(str).map(mapping)

print(train['layout_type'].head())
print(train['layout_type'].dtype)

['layout_type']
['layout_type']
0    0
1    0
2    0
3    0
4    0
Name: layout_type, dtype: int64
int64


## 3. 피처 및 타겟 설정

In [ ]:
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
print(f"피처 수: {len(feature_cols)}")

피처 수: 150


In [44]:
new_ratio_cols = [
    'order_per_robot',
    'charge_pressure',
    'congestion_per_robot',
    'orders_per_pack_station',
    'active_per_aisle_width',
    'congestion_per_aisle',
    'charging_per_robot',
    'low_battery_pressure',
    'pack_pressure',
    'intersection_pressure',
    'robot_density',
    'charger_per_robot',
    'emergency_complexity'
]

print(train[new_ratio_cols].head())
print(train[new_ratio_cols].isnull().sum())

   order_per_robot  charge_pressure  congestion_per_robot  \
0         0.000000              0.0                   0.0   
1         4.823529              0.0                   0.0   
2         4.718750              0.0                   0.0   
3         4.764706              0.0                   0.0   
4         4.823529              0.0                   0.0   

   orders_per_pack_station  active_per_aisle_width  congestion_per_aisle  \
0                 0.000000                9.267241                   0.0   
1                13.666667                7.112069                   0.0   
2                12.583333                6.681034                   0.0   
3                13.500000                7.112069                   0.0   
4                13.666667                7.112069                   0.0   

   charging_per_robot  low_battery_pressure  pack_pressure  \
0                 0.0                   0.0       0.083333   
1                 0.0                   0.0       0.

## 4. 모델 학습 (GroupKFold)

### LightGBM

In [45]:
X = train[feature_cols]
y = train[TARGET]
groups = train['scenario_id']
X_test = test[feature_cols]

def run_lgb_cv(X, y, X_test, groups, use_log_target=False, params=None, n_splits=5):
    if params is None:
        params = {
            'n_estimators': 1000,
            'learning_rate': 0.05,
            'max_depth': 7,
            'num_leaves': 63,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'reg_alpha': 0.1,
            'reg_lambda': 0.1,
            'random_state': 42,
            'verbose': -1,
        }

    kf = GroupKFold(n_splits=n_splits)
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y, groups=groups)):
        print(f"── LightGBM Fold {fold + 1} ──")

        X_tr = X.iloc[train_idx]
        X_val = X.iloc[valid_idx]

        y_tr = y.iloc[train_idx]
        y_val = y.iloc[valid_idx]

        if use_log_target:
            y_tr_fit = np.log1p(y_tr)
            y_val_fit = np.log1p(y_val)
        else:
            y_tr_fit = y_tr
            y_val_fit = y_val

        model = LGBMRegressor(**params)

        model.fit(
            X_tr, y_tr_fit,
            eval_set=[(X_val, y_val_fit)],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
        )

        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)

        if use_log_target:
            val_pred = np.expm1(val_pred)
            test_pred = np.expm1(test_pred)

        val_pred = np.clip(val_pred, 0, None)
        test_pred = np.clip(test_pred, 0, None)

        oof_preds[valid_idx] = val_pred
        test_preds += test_pred / n_splits

    mae = mean_absolute_error(y, oof_preds)
    return oof_preds, test_preds, mae

In [46]:
# 원본 target 버전
oof_raw, test_raw, mae_raw = run_lgb_cv(
    X=X,
    y=y,
    X_test=X_test,
    groups=groups,
    use_log_target=False
)

print(f"[원본 target] OOF MAE: {mae_raw:.4f}")

# log1p target 버전
oof_log, test_log, mae_log = run_lgb_cv(
    X=X,
    y=y,
    X_test=X_test,
    groups=groups,
    use_log_target=True
)

print(f"[log1p target] OOF MAE: {mae_log:.4f}")

── LightGBM Fold 1 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 500.114
Early stopping, best iteration is:
[78]	valid_0's l2: 499.378
── LightGBM Fold 2 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 485.298
Early stopping, best iteration is:
[63]	valid_0's l2: 484.417
── LightGBM Fold 3 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 490.499
Early stopping, best iteration is:
[58]	valid_0's l2: 489.132
── LightGBM Fold 4 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 379.047
Early stopping, best iteration is:
[91]	valid_0's l2: 377.838
── LightGBM Fold 5 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 442.225
Early stopping, best iteration is:
[115]	valid_0's l2: 441.899
[원본 target] OOF MAE: 9.6055
── LightGBM Fold 1 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.4

### CatBoost

In [47]:
def run_cat_cv(X, y, X_test, groups, use_log_target=False, params=None, n_splits=5):
    if params is None:
        params = {
            'iterations': 1000,
            'learning_rate': 0.05,
            'depth': 8,
            'loss_function': 'MAE',
            'eval_metric': 'MAE',
            'random_seed': 42,
            'verbose': 100
        }

    kf = GroupKFold(n_splits=n_splits)
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y, groups=groups)):
        print(f"── CatBoost Fold {fold + 1} ──")

        X_tr = X.iloc[train_idx]
        X_val = X.iloc[valid_idx]

        y_tr = y.iloc[train_idx]
        y_val = y.iloc[valid_idx]

        if use_log_target:
            y_tr_fit = np.log1p(y_tr)
            y_val_fit = np.log1p(y_val)
        else:
            y_tr_fit = y_tr
            y_val_fit = y_val

        model = CatBoostRegressor(**params)

        model.fit(
            X_tr, y_tr_fit,
            eval_set=(X_val, y_val_fit),
            use_best_model=True,
            early_stopping_rounds=50
        )

        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)

        if use_log_target:
            val_pred = np.expm1(val_pred)
            test_pred = np.expm1(test_pred)

        val_pred = np.clip(val_pred, 0, None)
        test_pred = np.clip(test_pred, 0, None)

        oof_preds[valid_idx] = val_pred
        test_preds += test_pred / n_splits

    mae = mean_absolute_error(y, oof_preds)
    return oof_preds, test_preds, mae

In [48]:
# CatBoost 원본 target
cat_oof_raw, cat_test_raw, cat_mae_raw = run_cat_cv(
    X=X,
    y=y,
    X_test=X_test,
    groups=groups,
    use_log_target=False
)

print(f"[CatBoost raw] OOF MAE: {cat_mae_raw:.4f}")

# CatBoost log1p target
cat_oof_log, cat_test_log, cat_mae_log = run_cat_cv(
    X=X,
    y=y,
    X_test=X_test,
    groups=groups,
    use_log_target=True
)

print(f"[CatBoost log1p] OOF MAE: {cat_mae_log:.4f}")

── CatBoost Fold 1 ──
0:	learn: 14.1740059	test: 14.2746406	best: 14.2746406 (0)	total: 60.3ms	remaining: 1m
100:	learn: 8.8735451	test: 9.2562231	best: 9.2562231 (100)	total: 5s	remaining: 44.5s
200:	learn: 8.5558854	test: 9.2037641	best: 9.2029655 (193)	total: 9.87s	remaining: 39.2s
300:	learn: 8.2891735	test: 9.1845266	best: 9.1819776 (272)	total: 14.7s	remaining: 34s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 9.181977624
bestIteration = 272

Shrink model to first 273 iterations.
── CatBoost Fold 2 ──
0:	learn: 14.2167612	test: 14.1246372	best: 14.1246372 (0)	total: 78.1ms	remaining: 1m 17s
100:	learn: 8.8978612	test: 9.1186847	best: 9.1186847 (100)	total: 4.75s	remaining: 42.3s
200:	learn: 8.5953586	test: 9.0934692	best: 9.0916216 (190)	total: 9.41s	remaining: 37.4s
300:	learn: 8.2979124	test: 9.0586523	best: 9.0579128 (295)	total: 14s	remaining: 32.5s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 9.055822993
bestIteration = 320

Shrink mo

## 5. 결과 확인

In [49]:
if mae_log < mae_raw:
    print("log1p target 버전 선택")
    oof_preds = oof_log
    test_preds = test_log
    best_mae = mae_log
    best_mode = 'log1p'
else:
    print("원본 target 버전 선택")
    oof_preds = oof_raw
    test_preds = test_raw
    best_mae = mae_raw
    best_mode = 'raw'

lgb_oof = oof_preds
lgb_test = test_preds
lgb_mae = best_mae
lgb_mode = best_mode

print(f"[LightGBM 최종] mode={lgb_mode}, OOF MAE={lgb_mae:.4f}")

log1p target 버전 선택
[LightGBM 최종] mode=log1p, OOF MAE=9.0447


In [50]:
if cat_mae_log < cat_mae_raw:
    cat_oof = cat_oof_log
    cat_test = cat_test_log
    cat_mae = cat_mae_log
    cat_mode = 'log1p'
else:
    cat_oof = cat_oof_raw
    cat_test = cat_test_raw
    cat_mae = cat_mae_raw
    cat_mode = 'raw'

print(f"[CatBoost 최종] mode={cat_mode}, OOF MAE={cat_mae:.4f}")

[CatBoost 최종] mode=log1p, OOF MAE=9.0278


In [51]:
ensemble_oof = (lgb_oof + cat_oof) / 2
ensemble_test = (lgb_test + cat_test) / 2
ensemble_mae = mean_absolute_error(y, ensemble_oof)

print(f"[앙상블] OOF MAE: {ensemble_mae:.4f}")

[앙상블] OOF MAE: 8.9969


In [52]:
results = {
    'lightgbm': lgb_mae,
    'catboost': cat_mae,
    'ensemble': ensemble_mae
}

best_model_name = min(results, key=results.get)
best_score = results[best_model_name]

print("모델별 OOF MAE:", results)
print(f"최종 선택: {best_model_name}, OOF MAE: {best_score:.4f}")

if best_model_name == 'lightgbm':
    final_test_preds = lgb_test
elif best_model_name == 'catboost':
    final_test_preds = cat_test
else:
    final_test_preds = ensemble_test

모델별 OOF MAE: {'lightgbm': 9.044667013741469, 'catboost': 9.027816045774944, 'ensemble': 8.996859631542158}
최종 선택: ensemble, OOF MAE: 8.9969


## 6. 제출 파일 생성

In [53]:
submission = pd.DataFrame({
    'ID': test['ID'],
    TARGET: final_test_preds
})

submission.to_csv('./submission.csv', index=False)
print("submission.csv 저장 완료.")

check = pd.read_csv('./submission.csv')
print(check.shape)
print(check.head())
print(check[TARGET].min(), check[TARGET].max())

submission.csv 저장 완료.
(50000, 2)
            ID  avg_delay_minutes_next_30m
0  TEST_015375                   18.373973
1  TEST_015376                   30.102329
2  TEST_015377                   32.053051
3  TEST_015378                   32.752850
4  TEST_015379                   37.224773
0.0443532946478498 60.356290756909694


In [54]:
check = pd.read_csv('./submission.csv')
print(check.shape)
print(check.head())
print(check['avg_delay_minutes_next_30m'].min(), check['avg_delay_minutes_next_30m'].max())

(50000, 2)
            ID  avg_delay_minutes_next_30m
0  TEST_015375                   18.373973
1  TEST_015376                   30.102329
2  TEST_015377                   32.053051
3  TEST_015378                   32.752850
4  TEST_015379                   37.224773
0.0443532946478498 60.356290756909694


In [55]:
import numpy as np

print("index 동일:", submission.index.equals(check.index))
print("ID 동일:", (submission['ID'] == check['ID']).all())
print("dtype 비교:")
print(submission.dtypes)
print(check.dtypes)

pred_col = 'avg_delay_minutes_next_30m'

diff = np.abs(submission[pred_col].values - check[pred_col].values)
print("allclose:", np.allclose(submission[pred_col].values, check[pred_col].values))
print("최대 차이:", diff.max())
print("차이 있는 행 수:", (diff > 0).sum())

index 동일: True
ID 동일: True
dtype 비교:
ID                             object
avg_delay_minutes_next_30m    float64
dtype: object
ID                             object
avg_delay_minutes_next_30m    float64
dtype: object
allclose: True
최대 차이: 7.105427357601002e-15
차이 있는 행 수: 6554


In [56]:
np.allclose(submission[pred_col], check[pred_col])

True